# Router 분류와 병렬 실행

한 번의 분류 결과로 서로 독립적인 전문 작업을 병렬 실행하면 호출 구조가 어떻게 달라질까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 구조화된 분류

### 코드 읽기와 비교 경계

`with_structured_output(Route)`는 모델의 응답을 자유로운 문장이 아니라 `Route` 스키마에 맞는 객체로 받도록 구성합니다. `destinations`는 `policy`와 `shipping`만 담을 수 있는 목록이므로 오타나 정의하지 않은 경로가 후속 실행으로 넘어가는 일을 줄입니다. `router.invoke(...)`가 실제 모델을 호출한 뒤 반환한 `Route` 객체의 `.destinations`를 읽어 실행할 함수를 정합니다. 스키마는 분류 결과의 형식을 검사하지만, 모델이 업무 의미에 맞는 분야를 골랐는지는 별도 평가가 필요합니다.

`ThreadPoolExecutor(max_workers=2)`는 두 전문가가 서로의 결과를 기다릴 필요가 없을 때 두 호출을 겹쳐 실행합니다. 환불 금액을 배송 조사 결과로 계산해야 한다면 먼저 배송 결과를 얻어야 하므로 병렬화하면 안 됩니다. `pool.map`은 입력 순서에 맞춰 결과를 반환하고 `zip`이 분야 이름과 답을 묶습니다. 병렬 실행은 개별 호출 수를 줄이지 않으며, 가장 느린 작업과 스레드 준비 시간을 포함한 전체 지연만 바꿀 수 있습니다.


In [2]:
# 모델이 요청에 필요한 전문 분야를 정해진 목록으로 반환하게 합니다.
from concurrent.futures import ThreadPoolExecutor
from typing import Literal
from pydantic import BaseModel
from langchain.chat_models import init_chat_model

class Route(BaseModel):
    destinations: list[Literal["policy", "shipping"]]

model = init_chat_model(f"openai:{MODEL}", reasoning_effort="none")
router = model.with_structured_output(Route)
request = "배송이 늦었습니다. 환불 규정과 현재 배송 안내를 함께 알려 주세요."
route = router.invoke("요청을 처리할 분야를 모두 고르세요: " + request)
print("선택된 분야:", route.destinations)


선택된 분야: ['policy', 'shipping']


## 선택된 전문가를 병렬 실행


In [3]:
# 서로 독립적인 전문 모델 호출을 최대 두 작업으로 동시에 실행합니다.
def run_specialist(name: str) -> str:
    prompts = {
        "policy": "환불 규정 확인 항목을 한 문장으로 답하세요: ",
        "shipping": "배송 상태 확인 항목을 한 문장으로 답하세요: ",
    }
    return model.invoke(prompts[name] + request).text

with ThreadPoolExecutor(max_workers=2) as pool:
    results = dict(zip(route.destinations, pool.map(run_specialist, route.destinations)))
for name, result in results.items():
    print(name, "→", result)
assert set(results) == set(route.destinations)


policy → 배송 지연으로 인한 환불은 상품별/판매처별 환불 규정에 따라 달라지며, 현재 배송 상태와 예정일은 주문 내역 또는 배송조회에서 확인해 주세요.
shipping → 배송 상태 확인: 현재 배송이 지연되고 있으며, 환불 규정은 상품 및 지연 사유에 따라 달라질 수 있으니 주문 내역 확인 후 고객센터에 문의해 현재 배송 안내와 함께 환불 가능 여부를 확인해 주세요.


## 병렬 결과 종합


In [4]:
# 분야 이름을 붙여 두 전문 결과를 하나의 근거 문자열로 합칩니다.
combined = "\n".join(f"[{name}] {value}" for name, value in results.items())
print(combined)
print("전문 결과 수:", len(results))


[policy] 배송 지연으로 인한 환불은 상품별/판매처별 환불 규정에 따라 달라지며, 현재 배송 상태와 예정일은 주문 내역 또는 배송조회에서 확인해 주세요.
[shipping] 배송 상태 확인: 현재 배송이 지연되고 있으며, 환불 규정은 상품 및 지연 사유에 따라 달라질 수 있으니 주문 내역 확인 후 고객센터에 문의해 현재 배송 안내와 함께 환불 가능 여부를 확인해 주세요.
전문 결과 수: 2
